In [ ]:
%run ../../commonsetup/Setup-Common

In [ ]:
SERVICE_DATE_EXPR = ("COALESCE(CAST(try_to_timestamp(start_date, 'yyyyMMdd') AS DATE), "
                     "CAST(from_utc_timestamp(poll_timestamp, 'Australia/Sydney') AS DATE))")
PASSENGER_SERVICE_EXPR = "(COALESCE(route_id, '') NOT LIKE 'RTTA%' AND COALESCE(trip_headsign, '') <> 'Empty Train')"


In [ ]:
MAX_PLAUSIBLE_DELAY_S = 21600
TRIPS_SEEN_THRESHOLD = 100
FULL_DAY_HOURS = 11

In [ ]:
base_view_query = f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_stop_latest_v AS
    WITH stop_rows AS (
    SELECT *,
        {SERVICE_DATE_EXPR} AS service_date
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_delays_enriched
    WHERE stop_id IS NOT NULL
        AND COALESCE(trip_schedule_relationship, 'SCHEDULED') = 'SCHEDULED'
        AND {PASSENGER_SERVICE_EXPR}
    ),
    ranked AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY gtfs_mode, service_date, trip_id, stop_id, COALESCE(stop_sequence, -1)
        ORDER BY poll_timestamp DESC) AS rn
    FROM stop_rows
    )
    SELECT gtfs_mode, service_date, trip_id, route_id, route_short_name, route_long_name,
        trip_headsign, direction_id, stop_sequence, stop_id, stop_name,
        arrival_delay_seconds, departure_delay_seconds,
        arrival_timestamp, departure_timestamp, poll_timestamp,
        ABS(COALESCE(arrival_delay_seconds, departure_delay_seconds)) > {MAX_PLAUSIBLE_DELAY_S} AS is_implausible_delay
    FROM ranked WHERE rn = 1
"""

In [ ]:
row_level_headways_query = f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_stop_headways_v AS
    WITH polled AS (
    SELECT DISTINCT
        CAST(from_utc_timestamp(poll_timestamp, 'Australia/Sydney') AS DATE) AS poll_date,
        hour(from_utc_timestamp(poll_timestamp, 'Australia/Sydney')) AS poll_hour
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_vehicle_positions
    ),
    ordered AS (
    SELECT b.gtfs_mode, b.service_date, b.route_id, b.route_short_name, b.stop_id, b.stop_name,
            b.trip_headsign, b.direction_id, b.trip_id, b.departure_timestamp,
            LAG(b.departure_timestamp) OVER (
            PARTITION BY b.gtfs_mode, b.service_date, b.stop_id
            ORDER BY b.departure_timestamp) AS prev_departure
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_stop_latest_v b
    JOIN polled p
        ON CAST(from_utc_timestamp(b.departure_timestamp, 'Australia/Sydney') AS DATE) = p.poll_date
        AND hour(from_utc_timestamp(b.departure_timestamp, 'Australia/Sydney')) = p.poll_hour
    WHERE b.departure_timestamp IS NOT NULL
    )
    SELECT *,
    ROUND((unix_timestamp(departure_timestamp) - unix_timestamp(prev_departure)) / 60.0, 2) AS headway_minutes,
    hour(from_utc_timestamp(departure_timestamp, 'Australia/Sydney')) AS local_hour
    FROM ordered
    WHERE prev_departure IS NOT NULL
        AND hour(from_utc_timestamp(prev_departure, 'Australia/Sydney'))
          = hour(from_utc_timestamp(departure_timestamp, 'Australia/Sydney'))
"""

##### Since only metro currently has a regular departure time 'Sydney Trains' departure times cover only ~36% of trips; headways would be biased
##### Hence, filtering only for 'metro' and exposed through `gold_headway_regularity_v`
##### `gold_stop_headways_v` covers all modes, so the unfiltered data is easy to find.

In [ ]:
regularity_summary_query = f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_headway_regularity_v AS
   SELECT h.gtfs_mode, h.service_date, h.local_hour,
       COALESCE(c.is_full_day, FALSE) AS is_full_day,
       h.stop_id, h.stop_name,
       COUNT(*) AS headway_count,
       ROUND(AVG(h.headway_minutes), 2) AS avg_headway_minutes,
       ROUND(STDDEV(h.headway_minutes), 2) AS stddev_headway_minutes,
       ROUND(STDDEV(h.headway_minutes) / NULLIF(AVG(h.headway_minutes), 0), 2) AS headway_cv
   FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_stop_headways_v h
   LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.gold_capture_coverage_v c ON h.service_date = c.local_date
   WHERE h.gtfs_mode = 'metro'
   GROUP BY ALL
"""

In [ ]:
route_delay_summaries_query = f"""
CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_route_delay_summary_v AS
WITH d AS (
  SELECT *, COALESCE(arrival_delay_seconds, departure_delay_seconds) AS delay_s,
    CASE WHEN route_short_name RLIKE '^T[0-9]$' THEN 'Suburban'
         WHEN route_short_name = 'M1' THEN 'Metro'
         WHEN route_short_name IS NULL THEN 'Unresolved'
         ELSE 'Intercity/Regional' END AS service_group
  FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_stop_latest_v
)
SELECT gtfs_mode, service_date, service_group, route_id,
  COALESCE(route_short_name, 'Unscheduled/Unresolved') AS route_short_name, route_long_name,
  hour(from_utc_timestamp(COALESCE(arrival_timestamp, departure_timestamp, poll_timestamp), 'Australia/Sydney')) AS local_hour,
  CASE WHEN COALESCE(arrival_timestamp, departure_timestamp) IS NULL THEN 'poll' ELSE 'scheduled' END AS hour_source,
  COUNT(*) AS stop_events,
  ROUND(AVG(delay_s)/60.0, 1) AS avg_delay_minutes,
  ROUND(percentile_approx(delay_s, 0.5)/60.0, 1) AS median_delay_minutes,
  ROUND(percentile_approx(delay_s, 0.9)/60.0, 1) AS p90_delay_minutes,
  ROUND(100.0 * AVG(CASE WHEN delay_s > 300 THEN 1 ELSE 0 END), 1) AS pct_over_5_min
FROM d WHERE delay_s IS NOT NULL AND NOT COALESCE(is_implausible_delay, FALSE)
GROUP BY ALL
"""

In [ ]:
vehicle_position_query = f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_vehicle_positions_current_v AS
    SELECT
      vp.gtfs_mode,
      vp.vehicle_id,
      vp.vehicle_label,
      vp.trip_id,
      COALESCE(vp.route_id, t.route_id) AS route_id,
      COALESCE(r.route_short_name, 'Unscheduled/Unresolved') AS route_short_name,
      r.route_long_name,
      t.trip_headsign, t.direction_id,
      vp.schedule_relationship, vp.latitude, vp.longitude, vp.position_timestamp,
      vp.stop_id, s.stop_name, vp.congestion_level,
      COALESCE(vp.occupancy_status, 'Not Reported') AS occupancy_status,
      vp.poll_timestamp
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_vehicle_positions_current vp
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_trips  t ON vp.gtfs_mode = t.gtfs_mode AND vp.trip_id = t.trip_id
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_routes r ON vp.gtfs_mode = r.gtfs_mode AND COALESCE(vp.route_id, t.route_id) = r.route_id
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_stops  s ON vp.gtfs_mode = s.gtfs_mode AND vp.stop_id = s.stop_id
"""

In [ ]:
station_map_query = f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_stop_station_v AS
    SELECT p.gtfs_mode, p.stop_id, p.stop_name AS platform_name,
        COALESCE(p.parent_station, p.stop_id) AS station_id,
        COALESCE(st.stop_name, p.stop_name) AS station_name
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_stops p
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_stops st
    ON p.gtfs_mode = st.gtfs_mode AND p.parent_station = st.stop_id
"""

In [ ]:
bronze_streaming_run_log_query=f"""
CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_capture_coverage_v AS
WITH runs AS (
  SELECT gtfs_mode,
         from_utc_timestamp(CAST(run_timestamp AS TIMESTAMP), 'Australia/Sydney') AS run_local,
         polls_attempted, polls_failed, rows_written
  FROM {CATALOG_NAME}.{SCHEMA_NAME}.bronze_streaming_run_log
  WHERE outcome = 'success'
)
SELECT CAST(run_local AS DATE) AS local_date,
       CASE WHEN dayofweek(run_local) IN (1, 7) THEN 'Weekend' ELSE 'Weekday' END AS day_type,
       COUNT(DISTINCT hour(run_local)) AS hours_captured,
       MIN(hour(run_local)) AS first_hour,
       MAX(hour(run_local)) AS last_hour,
       SUM(polls_attempted) AS polls_attempted,
       SUM(polls_failed) AS polls_failed,
       COUNT(DISTINCT hour(run_local)) >= {FULL_DAY_HOURS} AS is_full_day
FROM runs
GROUP BY ALL
"""

In [ ]:
queries = [base_view_query, row_level_headways_query, bronze_streaming_run_log_query, regularity_summary_query, route_delay_summaries_query, 
           vehicle_position_query, station_map_query]

for query in queries:
    spark.sql(query)

### Features

In [ ]:
service_profile_active_vehicles_query=f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_service_profile_v AS
    SELECT gtfs_mode,
        CAST(from_utc_timestamp(poll_timestamp, 'Australia/Sydney') AS DATE) AS local_date,
        CASE WHEN dayofweek(from_utc_timestamp(poll_timestamp, 'Australia/Sydney')) IN (1, 7)
                THEN 'Weekend' ELSE 'Weekday' END AS day_type,
        hour(from_utc_timestamp(poll_timestamp, 'Australia/Sydney')) AS local_hour,
        COUNT(DISTINCT vehicle_id) AS active_vehicles,
        COUNT(DISTINCT trip_id) AS active_trips
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_vehicle_positions
    WHERE vehicle_id <> 'Unassigned' AND trip_id IS NOT NULL
    GROUP BY ALL
"""

In [ ]:
delay_hotspots_query=f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_delay_hotspots_v AS
    SELECT b.gtfs_mode, b.service_date, m.station_id,
        COALESCE(m.station_name, b.stop_name, 'Unresolved') AS station_name,
        b.route_short_name,
        COUNT(*) AS observations,
        COUNT(DISTINCT b.trip_id) AS trips_observed,
        ROUND(AVG(b.d) / 60.0, 2) AS avg_delay_minutes,
        ROUND(percentile_approx(b.d, 0.9) / 60.0, 2) AS p90_delay_minutes,
        ROUND(100.0 * AVG(CASE WHEN b.d > 300 THEN 1 ELSE 0 END), 1) AS pct_over_5_min
    FROM (SELECT *, COALESCE(arrival_delay_seconds, departure_delay_seconds) AS d
        FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_stop_latest_v
        WHERE COALESCE(arrival_delay_seconds, departure_delay_seconds) IS NOT NULL) b
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.gold_stop_station_v m
    ON b.gtfs_mode = m.gtfs_mode AND b.stop_id = m.stop_id
    WHERE NOT COALESCE(is_implausible_delay, FALSE)
    GROUP BY ALL
"""

In [ ]:
cancel_replace_triplevel_query=f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_cancellation_summary_v AS
    WITH trips AS (
    SELECT gtfs_mode, {SERVICE_DATE_EXPR} AS service_date, trip_id,
            MAX(route_short_name) AS route_short_name,
            CASE WHEN MAX(CASE WHEN trip_schedule_relationship = 'CANCELED' THEN 1 ELSE 0 END) = 1 THEN 'CANCELED'
                WHEN MAX(CASE WHEN trip_schedule_relationship = 'REPLACEMENT' THEN 1 ELSE 0 END) = 1 THEN 'REPLACEMENT'
                ELSE 'SCHEDULED' END AS trip_status
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_delays_enriched
    WHERE {PASSENGER_SERVICE_EXPR}
    GROUP BY gtfs_mode, {SERVICE_DATE_EXPR}, trip_id
    )
    SELECT gtfs_mode, service_date, COALESCE(route_short_name, 'Unresolved') AS route_short_name,
        COUNT(*) AS trips_seen,
        SUM(CASE WHEN trip_status = 'CANCELED' THEN 1 ELSE 0 END) AS cancelled_trips,
        SUM(CASE WHEN trip_status = 'REPLACEMENT' THEN 1 ELSE 0 END) AS replacement_trips,
        ROUND(100.0 * SUM(CASE WHEN trip_status = 'CANCELED' THEN 1 ELSE 0 END) / COUNT(*), 1) AS cancellation_rate_pct,
        ROUND(100.0 * SUM(CASE WHEN trip_status = 'REPLACEMENT' THEN 1 ELSE 0 END) / COUNT(*), 1) AS replacement_rate_pct
    FROM trips GROUP BY gtfs_mode, service_date, COALESCE(route_short_name, 'Unresolved')
"""

In [ ]:
most_served_stations_query=f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_most_served_stations_v AS
    SELECT b.gtfs_mode, b.service_date, m.station_id,
        COALESCE(m.station_name, b.stop_name, 'Unresolved') AS station_name,
        COUNT(DISTINCT b.trip_id) AS trips_served
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_stop_latest_v b
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.gold_stop_station_v m
    ON b.gtfs_mode = m.gtfs_mode AND b.stop_id = m.stop_id
    WHERE {PASSENGER_SERVICE_EXPR}
    GROUP BY ALL
"""

In [ ]:
destinations_query=f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_destinations_v AS
    SELECT gtfs_mode, service_date, COALESCE(trip_headsign, 'Unknown') AS destination,
        COUNT(DISTINCT trip_id) AS trips
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_stop_latest_v
    GROUP BY ALL
"""

In [ ]:
data_trust_panel_query=f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_data_trust_v AS
    SELECT gtfs_mode, {SERVICE_DATE_EXPR} AS service_date,
        COUNT(*) AS stop_update_rows,
        COUNT(DISTINCT trip_id) AS trips,
        ROUND(100.0 * AVG(CASE WHEN COALESCE(arrival_delay_seconds, departure_delay_seconds) IS NOT NULL THEN 1 ELSE 0 END), 1) AS pct_with_delay,
        ROUND(100.0 * AVG(CASE WHEN arrival_timestamp IS NOT NULL THEN 1 ELSE 0 END), 1) AS pct_with_arrival_time,
        ROUND(100.0 * AVG(CASE WHEN stop_sequence IS NOT NULL THEN 1 ELSE 0 END), 1) AS pct_with_stop_sequence,
        ROUND(100.0 * AVG(CASE WHEN stop_name IS NULL THEN 1 ELSE 0 END), 1) AS pct_unresolved_stop,
        ROUND(100.0 * AVG(CASE WHEN route_short_name IS NULL THEN 1 ELSE 0 END), 1) AS pct_unresolved_route,
        ROUND(100.0 * AVG(CASE WHEN trip_schedule_relationship = 'REPLACEMENT' THEN 1 ELSE 0 END), 1) AS pct_replacement_rows,
        ROUND(100.0 * AVG(CASE WHEN {PASSENGER_SERVICE_EXPR} THEN 0 ELSE 1 END), 1) AS pct_non_passenger_rows,
        ROUND(100.0 * AVG(CASE WHEN {PASSENGER_SERVICE_EXPR} AND route_short_name IS NULL THEN 1 ELSE 0 END), 1) AS pct_unresolved_passenger_route,
        ROUND(100.0 * AVG(CASE WHEN ABS(COALESCE(arrival_delay_seconds, departure_delay_seconds)) > {MAX_PLAUSIBLE_DELAY_S} THEN 1 ELSE 0 END), 1) AS pct_implausible_delay
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_delays_enriched
    WHERE stop_id IS NOT NULL
    GROUP BY ALL
"""

In [ ]:

regime_view_metro_only_query = f"""
CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_network_regime_v AS
WITH d AS (
  SELECT b.gtfs_mode, b.service_date,
         COUNT(DISTINCT m.station_id) AS stations_served,
         COUNT(DISTINCT b.trip_id) AS trips_seen
  FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_stop_latest_v b
  LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.gold_stop_station_v m
    ON b.gtfs_mode = m.gtfs_mode AND b.stop_id = m.stop_id
  WHERE b.gtfs_mode = 'metro'
  GROUP BY b.gtfs_mode, b.service_date
),
reliable AS (
  SELECT * FROM d WHERE trips_seen >= {TRIPS_SEEN_THRESHOLD}
),
c AS (
  SELECT *, CASE WHEN stations_served <> LAG(stations_served)
         OVER (PARTITION BY gtfs_mode ORDER BY service_date) THEN 1 ELSE 0 END AS changed
  FROM reliable
)
SELECT d.gtfs_mode, d.service_date, d.stations_served, d.trips_seen,
       d.trips_seen >= {TRIPS_SEEN_THRESHOLD} AS is_reliable_day,
       SUM(c.changed) OVER (PARTITION BY c.gtfs_mode ORDER BY c.service_date) + 1 AS network_regime
FROM d LEFT JOIN c USING (gtfs_mode, service_date)
"""

In [ ]:
feature_queries = [
                    service_profile_active_vehicles_query,
                    delay_hotspots_query,
                    cancel_replace_triplevel_query,
                    most_served_stations_query,
                    destinations_query,
                    data_trust_panel_query,
                    regime_view_metro_only_query
                 ]

for feature_query in feature_queries:
    spark.sql(feature_query)